# 07 — Personalized Recommendation Engine

**Component 4 of the Explainable AI Decision Framework.**

Maps the **same severity-axis contributing factors** the explanation paragraph
was built from onto concrete, prioritized actions. Because both components
consume one shared `ExplanationFactor` list, a student never reads about one
set of pressures and then receives advice about a different set.

Rule-based and deterministic — not an LLM call, and not a second ML model. See
`docs/research/methodology.md` for the rationale.

### Scope boundary

This produces a recommendation for a **single point-in-time assessment**. The
Adaptive Recovery Framework (Component 5 — changing strategy when suggestions
are repeatedly ignored) is **not** implemented: it needs engagement history
across multiple check-ins, which requires backend user-history tables that do
not exist yet.

In [1]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split

from src.explainability import generate_explanation, severity_contributions
from src.recommendation import (
    MAX_RECOMMENDATIONS,
    MIN_SEVERITY_FOR_ACTION,
    RECOMMENDATION_CATALOGUE,
    build_recommendation_plan,
)

ARTIFACTS_DIR = Path("../artifacts")
DATA_PATH = "../datasets/raw/student_stress_factors.csv"
RANDOM_STATE, TEST_SIZE = 42, 0.2

print(f"{len(RECOMMENDATION_CATALOGUE)} recommendations defined")
print(f"Rules: raising factors only, severity >= {MIN_SEVERITY_FOR_ACTION}, "
      f"max {MAX_RECOMMENDATIONS} per plan")

14 recommendations defined
Rules: raising factors only, severity >= 0.02, max 3 per plan


## Reproduce the same model, split and four cases

In [2]:
model = joblib.load(ARTIFACTS_DIR / "stress_model_v2.pkl")
shap_config = json.loads((ARTIFACTS_DIR / "shap_config.json").read_text(encoding="utf-8"))

FEATURES = shap_config["feature_order"]
CLASS_LABELS = shap_config["target"]["classes"]
CLASS_MEANING = shap_config["target"]["class_meaning"]

df = pd.read_csv(DATA_PATH)
X, y = df[FEATURES], df[shap_config["target"]["name"]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
confidence = y_proba.max(axis=1)
correct = y_pred == y_test.values
shap_values = shap.TreeExplainer(model).shap_values(X_test)

selected = {}
for c in CLASS_LABELS:
    mask = correct & (y_test.values == c)
    if mask.any():
        selected[f"correct_class_{c}_{CLASS_MEANING[str(c)]}"] = int(
            np.where(mask)[0][np.argmax(confidence[mask])]
        )
w = np.where(~correct)[0]
worst = int(w[np.argmax(confidence[w])])
selected[f"MISCLASSIFIED_true{y_test.values[worst]}_pred{y_pred[worst]}"] = worst

print({k: v for k, v in selected.items()})

{'correct_class_0_low': 0, 'correct_class_1_moderate': 8, 'correct_class_2_high': 2, 'MISCLASSIFIED_true1_pred2': 29}


## What a student would actually receive

Explanation paragraph and recommendation plan together, as one message. The
technical block above each is for our verification only and never reaches the
student.

In [3]:
plans = {}

for name, idx in selected.items():
    contributions = severity_contributions(shap_values[idx], CLASS_LABELS)
    predicted_class = int(y_pred[idx])
    ctx = {
        "case": name,
        "test_row": idx,
        "true_class": int(y_test.values[idx]),
        "predicted_class": predicted_class,
        "model_confidence": float(confidence[idx]),
        "correct": bool(correct[idx]),
        "model_version": shap_config["model_version"],
    }

    explanation = generate_explanation(
        shap_values=contributions,
        feature_values=X_test.iloc[idx].values,
        feature_names=FEATURES,
        predicted_class=predicted_class,
        top_n=4,
        context=ctx,
    )
    # Same factor list drives both components.
    plan = build_recommendation_plan(
        factors=explanation.factors, predicted_class=predicted_class, context=ctx
    )
    plans[name] = (explanation, plan)

    print("#" * 78)
    print(f"# {name}   (row {idx})   true={y_test.values[idx]}  "
          f"pred={predicted_class}  confidence={confidence[idx]:.3f}")
    print("#" * 78)

    print("\n[ INTERNAL - severity contributions, never shown ]")
    print(pd.DataFrame(
        {"value": [f.feature_value for f in explanation.factors],
         "severity": [round(f.shap_value, 4) for f in explanation.factors],
         "direction": [f.direction for f in explanation.factors]},
        index=[f.feature for f in explanation.factors],
    ).to_string())

    print("\n" + "-" * 78)
    print("WHAT THE STUDENT SEES")
    print("-" * 78)
    print("\nYour check-in\n")
    print(explanation.user_facing())

    if plan.has_actions:
        print("\nSuggested next steps\n")
        for r in plan.recommendations:
            print(f"  {r.priority}. {r.title}")
            print(f"     {r.action}")
            print(f"     Why: {r.rationale}\n")
    else:
        print("\nSuggested next steps\n")
        print(f"  {plan.affirmation}\n")
    print()

##############################################################################
# correct_class_0_low   (row 0)   true=0  pred=0  confidence=1.000
##############################################################################

[ INTERNAL - severity contributions, never shown ]
                              value  severity direction
academic_performance            5.0   -0.1618    easing
basic_needs                     4.0   -0.1601    easing
teacher_student_relationship    5.0   -0.1504    easing
headache                        1.0   -0.1112    easing

------------------------------------------------------------------------------
WHAT THE STUDENT SEES
------------------------------------------------------------------------------

Your check-in

Your current wellbeing check-in suggests things feel reasonably steady for you at the moment. What does seem to be steadying things is that how your studies have been going seems to be working in your favour. It also helps that having your everyd

## Where the severity floor comes from

`MIN_SEVERITY_FOR_ACTION` should not be a number someone liked the look of.
This cell derives it from the observed distribution of |severity| across the
whole held-out test set, and — more importantly — measures whether it actually
does anything.

In [4]:
from src.recommendation import SEVERITY_PERCENTILE_BASIS

all_sev = np.stack([severity_contributions(shap_values[i], CLASS_LABELS)
                    for i in range(len(X_test))])
abs_sev = np.abs(all_sev).ravel()

print(f"|severity| over {all_sev.shape[0]} students x {all_sev.shape[1]} features "
      f"= {abs_sev.size} attributions\n")
for p in [10, 25, 50, 75, 90]:
    marker = "  <- basis for MIN_SEVERITY_FOR_ACTION" if p == SEVERITY_PERCENTILE_BASIS else ""
    print(f"  p{p:<3d} = {np.percentile(abs_sev, p):.4f}{marker}")

derived = np.percentile(abs_sev, SEVERITY_PERCENTILE_BASIS)
print(f"\np{SEVERITY_PERCENTILE_BASIS} = {derived:.4f}; "
      f"constant in code = {MIN_SEVERITY_FOR_ACTION}")

|severity| over 220 students x 14 features = 3080 attributions

  p10  = 0.0086
  p25  = 0.0200  <- basis for MIN_SEVERITY_FOR_ACTION
  p50  = 0.0498
  p75  = 0.0846
  p90  = 0.1331

p25 = 0.0200; constant in code = 0.02


In [5]:
# Does the floor ever actually exclude anything? Only factors that reach the
# top-4 selection can be filtered by it, so that is the population to check.
top4_idx = np.argsort(np.abs(all_sev), axis=1)[:, -4:]

raising_in_top4, below_floor = 0, 0
for i in range(len(X_test)):
    vals = [all_sev[i, j] for j in top4_idx[i] if all_sev[i, j] > 0]
    raising_in_top4 += len(vals)
    below_floor += sum(1 for v in vals if v < MIN_SEVERITY_FOR_ACTION)

smallest_top4 = np.sort(np.abs(all_sev), axis=1)[:, -4:].min()

print(f"Raising factors reaching the top-4 selection : {raising_in_top4}")
print(f"  of those below the {MIN_SEVERITY_FOR_ACTION} floor          : {below_floor} "
      f"({below_floor / max(raising_in_top4, 1):.1%})")
print(f"Smallest |severity| among ALL top-4 factors  : {smallest_top4:.4f}")
print()
print("The floor does not bind on this dataset. It is a guard rail for")
print("out-of-distribution input, not an active filter - see the constant's")
print("docstring in src/recommendation/engine.py. The affirmation seen for the")
print("low-stress case is caused by that student having no raising factors,")
print("not by this threshold.")

Raising factors reaching the top-4 selection : 430
  of those below the 0.02 floor          : 0 (0.0%)
Smallest |severity| among ALL top-4 factors  : 0.0346

The floor does not bind on this dataset. It is a guard rail for
out-of-distribution input, not an active filter - see the constant's
docstring in src/recommendation/engine.py. The affirmation seen for the
low-stress case is caused by that student having no raising factors,
not by this threshold.


## The low-stress case: when nothing warrants an action

A recommendation engine that always produces recommendations will invent
problems, which is exactly the "generic recommendation" failure `CLAUDE.md`
prohibits and would also corrupt the engagement signal the Adaptive Recovery
Framework will later depend on.

The engine therefore requires a factor to be **raising** stress *and* to exceed
a severity floor before it earns an action. When nothing qualifies it returns a
class-appropriate affirmation instead — acknowledging what is working rather
than manufacturing something to fix.

Note the class-2 affirmation deliberately differs: if a student is under high
pressure but no single factor stands out sharply enough to act on, the honest
response is to point toward a person, not an automated tip.

In [6]:
for name, (explanation, plan) in plans.items():
    raising = [f for f in explanation.factors if f.direction == "raising"]
    above = [f for f in raising if abs(f.shap_value) >= MIN_SEVERITY_FOR_ACTION]
    print(f"{name[:30]:32s} pred={plan.predicted_class}  "
          f"raising={len(raising)}/4  above_floor={len(above)}  "
          f"-> {'affirmation' if not plan.has_actions else str(len(plan.recommendations)) + ' action(s)'}")

correct_class_0_low              pred=0  raising=0/4  above_floor=0  -> affirmation
correct_class_1_moderate         pred=1  raising=2/4  above_floor=2  -> 2 action(s)
correct_class_2_high             pred=2  raising=4/4  above_floor=4  -> 3 action(s)
MISCLASSIFIED_true1_pred2        pred=2  raising=3/4  above_floor=3  -> 3 action(s)


## Coherence check: does the advice match what was explained?

Every recommended action must trace back to a factor the student was actually
told about. A suggestion about something the explanation never mentioned would
read as arbitrary and would break the faithfulness chain.

In [7]:
rows = []
for name, (explanation, plan) in plans.items():
    explained = {f.feature for f in explanation.factors}
    for r in plan.recommendations:
        rows.append({
            "case": name[:28],
            "recommended_for": r.feature,
            "priority": r.priority,
            "severity": round(r.severity_contribution, 4),
            "was_in_explanation": r.feature in explained,
            "category": r.category,
        })

if rows:
    coherence = pd.DataFrame(rows)
    print(coherence.to_string(index=False))
    print(f"\nRecommendations not traceable to the explanation: "
          f"{(~coherence['was_in_explanation']).sum()} of {len(coherence)}")
else:
    print("No recommendations generated across these cases.")

                     case              recommended_for  priority  severity  was_in_explanation          category
 correct_class_1_moderate teacher_student_relationship         1    0.0570                True          academic
 correct_class_1_moderate                  basic_needs         2    0.0523                True practical_support
     correct_class_2_high   extracurricular_activities         1    0.1437                True          academic
     correct_class_2_high                peer_pressure         2    0.1400                True            social
     correct_class_2_high                  self_esteem         3    0.1040                True   self_reflection
MISCLASSIFIED_true1_pred2                  self_esteem         1    0.1776                True   self_reflection
MISCLASSIFIED_true1_pred2         academic_performance         2    0.0964                True          academic
MISCLASSIFIED_true1_pred2               social_support         3    0.0857                True  

## Faithfulness log

In [8]:
out = Path("../experiments") / "recommendation_log_v2_local_cases.jsonl"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", encoding="utf-8") as f:
    for name, (_, plan) in plans.items():
        f.write(json.dumps(plan.faithfulness_record) + "\n")
print(f"Wrote {out}")

sample = plans[list(plans)[-1]][1].faithfulness_record
print("\nExample record (misclassified case):")
print(json.dumps({k: v for k, v in sample.items() if k != "recommendations"}, indent=2)[:1400])

Wrote ../experiments/recommendation_log_v2_local_cases.jsonl

Example record (misclassified case):
{
  "generated_utc": "2026-08-15T17:59:52Z",
  "component": "recommendation_engine",
  "scope": "single point-in-time assessment; no engagement history used",
  "adaptive_recovery_applied": false,
  "predicted_class": 2,
  "selection_rules": {
    "min_severity_for_action": 0.02,
    "max_recommendations": 3,
    "raising_factors_only": true
  },
  "candidate_factors": [
    {
      "feature": "self_esteem",
      "severity_contribution": 0.17757696532040057,
      "direction": "raising",
      "qualified": true
    },
    {
      "feature": "breathing_problem",
      "severity_contribution": -0.1444700748148095,
      "direction": "easing",
      "qualified": false
    },
    {
      "feature": "academic_performance",
      "severity_contribution": 0.0964115733320012,
      "direction": "raising",
      "qualified": true
    },
    {
      "feature": "social_support",
      "severity_con